# 03 — Feature Engineering

The baseline got 0.7829 AUC off the raw static columns. It only sees absolute amounts no
sense of whether a loan is affordable for this particular applicant, and no history at all.

Two ideas to fix that: ratios (payment burden, debt-to-income, utilization, a tree can't
divide two columns on its own), and collapsing the depth-1 tables down to one row per case
so the model gets to see previous applications, household, cards and deposits.

Split and metric are imported from `credit_risk`, so nothing changes vs notebook 02 except
the features.


In [1]:
import polars as pl
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from pathlib import Path

from credit_risk import gini_stability, temporal_train_val_split, select_numeric_features

DATA = Path("../data/parquet_files")
TRAIN = DATA / "train"

base = pl.read_parquet(TRAIN / "train_base.parquet")

static = pl.concat([
    pl.read_parquet(TRAIN / "train_static_0_0.parquet"),
    pl.read_parquet(TRAIN / "train_static_0_1.parquet"),
], how="vertical_relaxed")

print(f"Base:   {base.shape}")
print(f"Static: {static.shape}")

Base:   (1526659, 5)
Static: (1526659, 168)


## 1. Baseline feature matrix

Same 53 columns as notebook 02.


In [2]:
baseline_cols = select_numeric_features(static, suffixes=("A", "P"))
print(f"Baseline features: {len(baseline_cols)}")

df = base.join(static.select(["case_id"] + baseline_cols), on="case_id", how="left")

print(f"Joined shape:  {df.shape}")
print(f"Default rate:  {df['target'].mean():.2%}")

Baseline features: 53


Joined shape:  (1526659, 58)
Default rate:  3.14%


## 2. Ratio features

Ratios from the static amounts. `safe_ratio` nulls out anything with a null or non-positive
denominator so LightGBM is safe from an inf.


In [3]:
def safe_ratio(num: str, den: str) -> pl.Expr:
    """num / den, returning null when the denominator is null or <= 0."""
    return (
        pl.when(pl.col(den).is_null() | (pl.col(den) <= 0))
        .then(None)
        .otherwise(pl.col(num) / pl.col(den))
        .cast(pl.Float64)
    )


ratio_exprs = {
    # Payment burden
    "ratio_annuity_to_credit":      safe_ratio("annuity_780A", "credamount_770A"),
    "ratio_annuity_to_income":      safe_ratio("annuity_780A", "maininc_215A"),
    "ratio_annuity_to_maxannuity":  safe_ratio("annuity_780A", "maxannuity_159A"),
    # Loan size vs capacity
    "ratio_credit_to_income":       safe_ratio("credamount_770A", "maininc_215A"),
    "ratio_credit_to_price":        safe_ratio("credamount_770A", "price_1097A"),
    "ratio_disbursed_to_credit":    safe_ratio("disbursedcredamount_1113A", "credamount_770A"),
    "ratio_downpmt_to_credit":      safe_ratio("downpmt_116A", "credamount_770A"),
    "ratio_lastappr_to_credit":     safe_ratio("lastapprcredamount_781A", "credamount_770A"),
    # Existing leverage
    "ratio_currdebt_to_income":     safe_ratio("currdebt_22A", "maininc_215A"),
    "ratio_totaldebt_to_income":    safe_ratio("totaldebt_9A", "maininc_215A"),
    "ratio_debt_utilization":       safe_ratio("currdebt_22A", "totaldebt_9A"),
    "ratio_outstanding_to_credit":  safe_ratio("sumoutstandtotal_3546847A", "credamount_770A"),
}

df = df.with_columns([e.alias(name) for name, e in ratio_exprs.items()])
ratio_cols = list(ratio_exprs)

print(f"Ratio features: {len(ratio_cols)}")
print(
    df.select(ratio_cols)
    .describe()
    .select(["statistic"] + ratio_cols[:4])
)

Ratio features: 12
shape: (9, 5)
┌────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┐
│ statistic  ┆ ratio_annuity_to_cr ┆ ratio_annuity_to_in ┆ ratio_annuity_to_m ┆ ratio_credit_to_in │
│ ---        ┆ edit                ┆ come                ┆ axannuity          ┆ come               │
│ str        ┆ ---                 ┆ ---                 ┆ ---                ┆ ---                │
│            ┆ f64                 ┆ f64                 ┆ f64                ┆ f64                │
╞════════════╪═════════════════════╪═════════════════════╪════════════════════╪════════════════════╡
│ count      ┆ 1.526659e6          ┆ 1.015397e6          ┆ 1.071623e6         ┆ 1.015397e6         │
│ null_count ┆ 0.0                 ┆ 511262.0            ┆ 455036.0           ┆ 511262.0           │
│ mean       ┆ 0.098403            ┆ 0.77679             ┆ 0.507376           ┆ 9.28345            │
│ std        ┆ 0.045758            ┆ 140.567042          ┆

## 3. Depth-1 aggregations

These tables have multiple rows per `case_id`, so everything gets collapsed to one row and
left-joined. 

### 3a. Previous applications (`applprev_1`)

Only reading the columns I actually aggregate.


In [4]:
applprev_cols = [
    "case_id",
    "credamount_590A",         # size of the previous loan
    "annuity_853A",            # its instalment
    "actualdpd_943P",          # days past due actually observed
    "maxdpdtolerance_577P",    # worst DPD within tolerance
    "outstandingdebt_522A",
    "mainoccupationinc_437A",  # income stated at the time
    "tenor_203L",
    "downpmt_134A",
    "status_219L",             # application outcome code
    "rejectreason_755M",       # 'a55475b1' is this dataset's null hash
    "creationdate_885D",
]

applprev = pl.concat([
    pl.read_parquet(TRAIN / "train_applprev_1_0.parquet", columns=applprev_cols),
    pl.read_parquet(TRAIN / "train_applprev_1_1.parquet", columns=applprev_cols),
], how="vertical_relaxed")

print(f"applprev_1 rows: {len(applprev):,}")
print(f"cases covered:   {applprev['case_id'].n_unique():,} "
      f"({applprev['case_id'].n_unique() / len(base):.1%} of base)")
print(applprev["status_219L"].value_counts().sort("count", descending=True).head(6))

applprev_1 rows: 6,525,979
cases covered:   1,221,522 (80.0% of base)
shape: (6, 2)
┌─────────────┬─────────┐
│ status_219L ┆ count   │
│ ---         ┆ ---     │
│ str         ┆ u32     │
╞═════════════╪═════════╡
│ D           ┆ 2667953 │
│ K           ┆ 2658434 │
│ A           ┆ 715907  │
│ T           ┆ 441632  │
│ N           ┆ 30430   │
│ Q           ┆ 7477    │
└─────────────┴─────────┘


In [5]:
NULL_HASH = "a55475b1"  # placeholder value used for missing categorical hashes

applprev_agg = (
    applprev
    .with_columns(pl.col("creationdate_885D").str.to_date("%Y-%m-%d"))
    .group_by("case_id")
    .agg([
        pl.len().alias("prev_n_apps"),
        pl.col("credamount_590A").mean().alias("prev_credamount_mean"),
        pl.col("credamount_590A").max().alias("prev_credamount_max"),
        pl.col("credamount_590A").sum().alias("prev_credamount_sum"),
        pl.col("annuity_853A").mean().alias("prev_annuity_mean"),
        pl.col("annuity_853A").max().alias("prev_annuity_max"),
        pl.col("actualdpd_943P").mean().alias("prev_actualdpd_mean"),
        pl.col("actualdpd_943P").max().alias("prev_actualdpd_max"),
        pl.col("maxdpdtolerance_577P").mean().alias("prev_maxdpdtol_mean"),
        pl.col("maxdpdtolerance_577P").max().alias("prev_maxdpdtol_max"),
        (pl.col("actualdpd_943P") > 0).sum().alias("prev_n_dpd_events"),
        pl.col("outstandingdebt_522A").sum().alias("prev_outstanding_sum"),
        pl.col("mainoccupationinc_437A").max().alias("prev_income_max"),
        pl.col("tenor_203L").mean().alias("prev_tenor_mean"),
        pl.col("downpmt_134A").mean().alias("prev_downpmt_mean"),
        (pl.col("status_219L") == "A").sum().alias("prev_n_approved"),
        (pl.col("rejectreason_755M") != NULL_HASH).mean().alias("prev_reject_rate"),
        pl.col("creationdate_885D").max().alias("prev_last_date"),
        pl.col("creationdate_885D").min().alias("prev_first_date"),
    ])
)

print(f"applprev_agg: {applprev_agg.shape}")
del applprev
applprev_agg.head(3)

applprev_agg: (1221522, 20)


case_id,prev_n_apps,prev_credamount_mean,prev_credamount_max,prev_credamount_sum,prev_annuity_mean,prev_annuity_max,prev_actualdpd_mean,prev_actualdpd_max,prev_maxdpdtol_mean,prev_maxdpdtol_max,prev_n_dpd_events,prev_outstanding_sum,prev_income_max,prev_tenor_mean,prev_downpmt_mean,prev_n_approved,prev_reject_rate,prev_last_date,prev_first_date
i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f64,f64,f64,f64,u32,f64,date,date
1635096,8,27673.142857,60000.0,193712.0,1717.171429,3509.0,0.0,0.0,0.0,0.0,0,0.0,84000.0,26.571429,0.0,0,0.375,2019-07-23,2007-08-28
1679011,10,82641.377778,157000.0,743772.4,5037.000033,8962.8,0.0,0.0,0.0,0.0,0,0.0,66600.0,29.333333,0.0,0,0.5,2019-09-27,2012-03-06
1331118,18,46270.875,100000.0,740334.0,5175.412525,10929.0,0.0,0.0,0.076923,1.0,0,145204.205,50000.0,13.5,133.75,2,0.444444,2019-03-29,2012-01-24


### 3b. Person, debit card and deposit tables

`person_1` is one row per household member; `num_group1 == 0` is the applicant, so applicant
income and household income become separate features. Cards and deposits are thin tables,
just counts and balance/turnover summaries.


In [6]:
person = pl.read_parquet(
    TRAIN / "train_person_1.parquet",
    columns=["case_id", "num_group1", "mainoccupationinc_384A", "childnum_185L"],
)

person_agg = (
    person
    .group_by("case_id")
    .agg([
        pl.len().alias("person_n_records"),
        pl.col("mainoccupationinc_384A")
          .filter(pl.col("num_group1") == 0).first().alias("person_income_applicant"),
        pl.col("mainoccupationinc_384A").max().alias("person_income_max"),
        pl.col("mainoccupationinc_384A").sum().alias("person_income_sum"),
        pl.col("childnum_185L").max().alias("person_childnum_max"),
    ])
)

debitcard = pl.read_parquet(TRAIN / "train_debitcard_1.parquet")
debitcard_agg = (
    debitcard
    .group_by("case_id")
    .agg([
        pl.len().alias("debitcard_n"),
        pl.col("last180dayaveragebalance_704A").mean().alias("debitcard_avgbal_mean"),
        pl.col("last180dayturnover_1134A").sum().alias("debitcard_turnover180_sum"),
        pl.col("last30dayturnover_651A").sum().alias("debitcard_turnover30_sum"),
    ])
)

deposit = pl.read_parquet(TRAIN / "train_deposit_1.parquet")
deposit_agg = (
    deposit
    .group_by("case_id")
    .agg([
        pl.len().alias("deposit_n"),
        pl.col("amount_416A").sum().alias("deposit_amount_sum"),
        pl.col("amount_416A").max().alias("deposit_amount_max"),
    ])
)

for name, tbl in [("person", person_agg), ("debitcard", debitcard_agg), ("deposit", deposit_agg)]:
    print(f"{name:<10} {tbl.shape[0]:>9,} cases  ({tbl.shape[0] / len(base):>5.1%} coverage), "
          f"{tbl.shape[1] - 1} features")

del person, debitcard, deposit

person     1,526,659 cases  (100.0% coverage), 5 features
debitcard    111,772 cases  ( 7.3% coverage), 4 features
deposit      105,111 cases  ( 6.9% coverage), 3 features


### 3c. Recency and cross-table features

Days since the last previous application, how long the application history is, and the
requested amount vs the income declared on the person table.


In [7]:
df = (
    df
    .join(applprev_agg, on="case_id", how="left")
    .join(person_agg, on="case_id", how="left")
    .join(debitcard_agg, on="case_id", how="left")
    .join(deposit_agg, on="case_id", how="left")
    .with_columns(pl.col("date_decision").str.to_date("%Y-%m-%d").alias("_decision_date"))
    .with_columns([
        (pl.col("_decision_date") - pl.col("prev_last_date")).dt.total_days()
            .alias("prev_days_since_last"),
        (pl.col("_decision_date") - pl.col("prev_first_date")).dt.total_days()
            .alias("prev_history_days"),
        safe_ratio("credamount_770A", "person_income_applicant")
            .alias("ratio_credit_to_person_income"),
    ])
    .drop(["prev_last_date", "prev_first_date", "_decision_date"])
)

agg_cols = (
    [c for c in applprev_agg.columns if c not in ("case_id", "prev_last_date", "prev_first_date")]
    + [c for c in person_agg.columns if c != "case_id"]
    + [c for c in debitcard_agg.columns if c != "case_id"]
    + [c for c in deposit_agg.columns if c != "case_id"]
    + ["prev_days_since_last", "prev_history_days", "ratio_credit_to_person_income"]
)

engineered_cols = ratio_cols + agg_cols
feature_cols = baseline_cols + engineered_cols

print(f"Baseline features:   {len(baseline_cols)}")
print(f"  ratio features:    {len(ratio_cols) + 1}")
print(f"  depth-1 aggregates:{len(agg_cols) - 1}")
print(f"Engineered features: {len(engineered_cols)}")
print(f"TOTAL features:      {len(feature_cols)}")
print(f"Frame shape:         {df.shape}")

Baseline features:   53
  ratio features:    13
  depth-1 aggregates:31
Engineered features: 44
TOTAL features:      97
Frame shape:         (1526659, 102)


## 4. Training

Identical params and split to the baseline. Only the features change.


In [8]:
train_df, val_df = temporal_train_val_split(df, train_frac=0.8, week_col="WEEK_NUM")

print(f"Train weeks: {train_df['WEEK_NUM'].min()} to {train_df['WEEK_NUM'].max()} "
      f"({train_df['WEEK_NUM'].n_unique()} weeks, {len(train_df):,} rows)")
print(f"Val weeks:   {val_df['WEEK_NUM'].min()} to {val_df['WEEK_NUM'].max()} "
      f"({val_df['WEEK_NUM'].n_unique()} weeks, {len(val_df):,} rows)")

X_train = train_df.select(feature_cols).to_pandas()
y_train = train_df["target"].to_pandas()
X_val = val_df.select(feature_cols).to_pandas()
y_val = val_df["target"].to_pandas()

val_base = val_df.select(["WEEK_NUM", "target"]).to_pandas()

print(f"\nX_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")

Train weeks: 0 to 72 (73 weeks, 1,323,314 rows)
Val weeks:   73 to 91 (19 weeks, 203,345 rows)



X_train: (1323314, 97)
X_val:   (203345, 97)


In [9]:
params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "n_estimators": 1000,
    "is_unbalance": True,
    "verbosity": -1,
    "random_state": 42,
}

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_val],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100),
    ],
)

Training until validation scores don't improve for 50 rounds


[100]	valid_0's auc: 0.800805


[200]	valid_0's auc: 0.805487


[300]	valid_0's auc: 0.806188


Early stopping, best iteration is:
[275]	valid_0's auc: 0.806331


## 5. Evaluation

In [10]:
val_preds = model.predict(X_val, num_iteration=model.best_iteration)

auc = roc_auc_score(y_val, val_preds)
gini = 2 * auc - 1
stability = gini_stability(val_base, val_preds)

BASELINE = {
    "auc": 0.7829,
    "gini": 0.5658,
    "stability_score": 0.5367,
    "slope": 0.0042,
    "residual_std": 0.0411,
}

results = pd.DataFrame({
    "baseline": [BASELINE["auc"], BASELINE["gini"], BASELINE["stability_score"],
                 BASELINE["slope"], BASELINE["residual_std"]],
    "engineered": [auc, gini, stability["stability_score"],
                   stability["slope"], stability["residual_std"]],
}, index=["val_auc", "val_gini", "stability_score", "weekly_gini_slope", "residual_std"])
results["delta"] = results["engineered"] - results["baseline"]

print(f"Best iteration: {model.best_iteration}\n")
print(results.round(4).to_string())

Best iteration: 275

                   baseline  engineered   delta
val_auc              0.7829      0.8063  0.0234
val_gini             0.5658      0.6127  0.0469
stability_score      0.5367      0.5868  0.0501
weekly_gini_slope    0.0042      0.0035 -0.0007
residual_std         0.0411      0.0402 -0.0009


## 6. Feature importance

Top 20 by gain, engineered features flagged.


In [11]:
importance = (
    pd.DataFrame({
        "feature": model.feature_name(),
        "gain": model.feature_importance(importance_type="gain"),
    })
    .sort_values("gain", ascending=False)
    .reset_index(drop=True)
)
importance["engineered"] = importance["feature"].isin(engineered_cols)
importance["gain_share"] = importance["gain"] / importance["gain"].sum()

top20 = importance.head(20).copy()
top20["rank"] = top20.index + 1
print(top20[["rank", "feature", "gain", "gain_share", "engineered"]].to_string(index=False))

n_eng_top20 = int(top20["engineered"].sum())
eng_gain_share = importance.loc[importance["engineered"], "gain"].sum() / importance["gain"].sum()

print(f"\nEngineered features in the top 20: {n_eng_top20}")
print(f"Share of total gain from engineered features: {eng_gain_share:.1%} "
      f"(from {len(engineered_cols)}/{len(feature_cols)} = "
      f"{len(engineered_cols) / len(feature_cols):.1%} of columns)")
print("\nTop engineered features:")
print(
    importance[importance["engineered"]]
    .head(10)
    .assign(rank=lambda d: d.index + 1)[["rank", "feature", "gain", "gain_share"]]
    .to_string(index=False)
)

 rank                      feature         gain  gain_share  engineered
    1  avgdpdtolclosure24_3658938P 1.315489e+06    0.161298       False
    2      ratio_annuity_to_credit 9.286918e+05    0.113871        True
    3             prev_reject_rate 8.452513e+05    0.103640        True
    4          prev_maxdpdtol_mean 5.972344e+05    0.073230        True
    5            totalsettled_863A 4.512326e+05    0.055328       False
    6                  price_1097A 3.806755e+05    0.046676       False
    7         prev_days_since_last 2.506977e+05    0.030739        True
    8        ratio_credit_to_price 2.389138e+05    0.029294        True
    9              prev_tenor_mean 1.815137e+05    0.022256        True
   10    avgdbddpdlast24m_3658932P 1.545820e+05    0.018954       False
   11   inittransactionamount_650A 1.382512e+05    0.016952       False
   12            prev_history_days 1.254778e+05    0.015385        True
   13 maxdbddpdtollast12m_3658940P 1.148222e+05    0.014079     

## Results

| Metric | Baseline | Engineered | Δ |
|---|---|---|---|
| Val AUC | 0.7829 | 0.8063 | +0.0234 |
| Val Gini | 0.5658 | 0.6127 | +0.0469 |
| Stability score | 0.5367 | 0.5868 | +0.0501 |
| Weekly Gini slope | +0.0042 | +0.0035 | −0.0007 |
| Residual std | 0.0411 | 0.0402 | −0.0009 |

+2.3 AUC points and +5 stability points for 44 features. Eight of the top 20 features are engineered ones; the annuity/credit ratio and the reject rate from previous applications sit at #2 and #3, right behind the best raw DPD column. Slope and residual std barely move, so the lift holds up across the validation weeks instead of being concentrated
in a few of them.
